[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/logicmojo-ai-july-2026/blob/main/modules/module-4-llms-genai/01-working-with-llms/code/working_with_llms.ipynb)

# Class 4.1: Working with LLMs

Drive an LLM like an engineer. We will build a message list with a deliberate system prompt, estimate what a call costs, route one wrapper across three real providers, and log every call. These cells make REAL model calls, so first set up a provider: run Ollama locally (free) or put a Groq or Gemini API key in a .env file at the project root (see .env.example). The default provider is Ollama.

## Setup

New libraries for this class (numpy, pandas, requests came earlier; nothing else is needed for the offline cells):

```
pip install ollama groq google-genai python-dotenv streamlit
```

Install the library for the provider you will use. The simplest free option is Ollama running locally; Groq and Gemini need an API key in .env.

## 1. A call is a list of messages

In [1]:
# A chat request is an ordered list of role-tagged messages, not one string.
messages = [
    {"role": "system", "content": "You are a terse assistant for an accounting app. One short paragraph, no filler."},
    {"role": "user", "content": "What is an invoice?"},
]

for m in messages:
    print(f"{m['role']:>9} | {m['content']}")

# The system message sets the job once; the user message is this turn's request.
# In a multi-turn chat you also append the model's past {"role": "assistant", ...}
# turns, because the model is stateless: the list is its only memory.

   system | You are a terse assistant for an accounting app. One short paragraph, no filler.
     user | What is an invoice?


## 2. Estimate tokens, then cost

In [2]:
# Models read TOKENS (word pieces), not words, and both cost and the context
# window are counted in tokens. Providers report exact token counts, but offline
# we estimate: English averages about 1.3 tokens per word.
def estimate_tokens(text):
    # text.split() splits on spaces into words; times 1.3 approximates tokens.
    # max(1, ...) makes sure even an empty string counts as at least one token.
    return max(1, round(len(text.split()) * 1.3))

# Join every message's text into one string, then estimate its token count.
prompt = " ".join(m["content"] for m in messages)
print("approx input tokens:", estimate_tokens(prompt))

approx input tokens: 23


In [3]:
# A call is priced per token, and input tokens and output tokens are billed at
# DIFFERENT rates. We group providers into three price "tiers". Each entry is
# (price_per_input_token, price_per_output_token) in dollars per 1,000,000 tokens.
# These numbers are ILLUSTRATIVE and change often; the ratio between tiers is the
# lesson, not the exact figure.
PRICES = {                     # dollars per 1,000,000 tokens (input, output)
    "local":    (0.0,  0.0),   # runs on your machine: free
    "hosted":   (0.05, 0.08),  # hosted open models (Groq, Gemini flash-lite): cheap
    "frontier": (2.50, 10.0),  # top-end proprietary models: much pricier
}

def call_cost(n_in, n_out, tier):
    p_in, p_out = PRICES[tier]                 # unpack the two prices for this tier
    # divide token counts by 1e6 because prices are quoted per million tokens
    return n_in / 1e6 * p_in + n_out / 1e6 * p_out

# Price the same call (1500 input, 500 output tokens) at each tier to compare.
n_in, n_out = 1500, 500
for tier in PRICES:
    print(f"{tier:>9}: ${call_cost(n_in, n_out, tier):.6f}")

# Expected: local $0.000000, hosted ~$0.000115, frontier ~$0.008750.

    local: $0.000000
   hosted: $0.000115
 frontier: $0.008750


## 3. One wrapper, three providers

In [8]:
# The three real providers: ollama (local), groq (hosted), gemini (hosted cloud).
# Two beginner points to notice:
#  1. Each library is imported INSIDE its function, so you only need the one you use.
#  2. Keys are read from a .env file at the project root (see .env.example), never
#     hardcoded. load_dotenv copies those values into os.environ.
import os
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv())          # loads GROQ_API_KEY, GEMINI_API_KEY, etc.
except Exception:
    pass

def ollama_provider(messages, model="llama3.2"):
    import ollama                                   # pip install ollama; needs a local server
    r = ollama.chat(model=model, messages=messages)
    # Ollama returns the reply plus how many tokens it read and wrote.
    return {"text": r["message"]["content"],
            "in_tokens": r.get("prompt_eval_count", 0), "out_tokens": r.get("eval_count", 0)}

def groq_provider(messages, model="openai/gpt-oss-20b"):
    from groq import Groq                           # pip install groq; needs GROQ_API_KEY
    client = Groq(api_key=os.environ["GROQ_API_KEY"])
    r = client.chat.completions.create(model=model, messages=messages)
    u = r.usage                                     # exact token counts from the API
    return {"text": r.choices[0].message.content, "in_tokens": u.prompt_tokens, "out_tokens": u.completion_tokens}

def gemini_provider(messages, model="gemini-3.5-flash-lite"):
    from google import genai                        # pip install google-genai; needs GEMINI_API_KEY
    from google.genai import types
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    # Gemini wants the system prompt separately and the rest as one text blob.
    system = "\n".join(m["content"] for m in messages if m["role"] == "system")
    convo = "\n\n".join(f"{m['role']}: {m['content']}" for m in messages if m["role"] != "system") or " "
    r = client.models.generate_content(
        model=model, contents=convo,
        config=types.GenerateContentConfig(system_instruction=system or None))
    u = r.usage_metadata                            # Gemini's token usage lives here
    return {"text": r.text or "", "in_tokens": u.prompt_token_count, "out_tokens": u.candidates_token_count}

# A name -> function lookup so we can pick a provider by string.
PROVIDERS = {"ollama": ollama_provider, "groq": groq_provider, "gemini": gemini_provider}

def chat(messages, provider="ollama", **kw):
    # One entry point: look up the provider function and call it. This is a REAL
    # model call. If the provider is not set up you will get a clear error, which
    # is the point: no fake output.
    return PROVIDERS[provider](messages, **kw)

# A real call. Default is ollama; switch to "groq" or "gemini" if that is what you
# configured in .env. Requires the chosen provider to be available.
reply = chat(messages, provider="ollama")
print(reply["text"])
print(f"(in={reply['in_tokens']} out={reply['out_tokens']} tokens)")

# The rest of the course uses the slim, packaged version of this wrapper in llm.py
# (see the note near the end). Never hardcode keys; keep them in .env (gitignored).

model='llama3.2' created_at='2026-08-31T15:40:29.4558911Z' done=True done_reason='stop' total_duration=1035203300 load_duration=195806200 prompt_eval_count=47 prompt_eval_duration=251373000 eval_count=39 eval_duration=584042000 message=Message(role='assistant', content='An invoice is a document sent by a supplier or service provider to a customer stating the amount owed for goods or services provided, including the date of delivery, quantities sold, and total cost.', thinking=None, images=None, tool_name=None, tool_calls=None) logprobs=None
An invoice is a document sent by a supplier or service provider to a customer stating the amount owed for goods or services provided, including the date of delivery, quantities sold, and total cost.
(in=47 out=39 tokens)


In [9]:
reply = chat(messages, provider="groq")
print(reply["text"])
print(f"(in={reply['in_tokens']} out={reply['out_tokens']} tokens)")

An invoice is a commercial document issued by a seller to a buyer that itemizes goods or services provided, specifies quantities, prices, applicable taxes, and the total amount due, along with payment terms and deadlines. It serves as both a sales receipt and a request for payment, and is used for accounting, tax reporting, and cash‑flow management.
(in=96 out=124 tokens)


## 4. Log every call

In [10]:
import logging, time

# Configure logging ONCE for the whole notebook: show INFO and above, with a
# timestamp, level, and message on each line.
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("llm")

# Which price tier each provider belongs to (used to compute cost per call).
TIER_OF = {"ollama": "local", "groq": "hosted", "gemini": "hosted"}

def chat_logged(messages, provider="ollama", **kw):
    # Wrap the plain chat() call to also time it, price it, and log one line.
    t0 = time.perf_counter()                        # stopwatch start
    out = chat(messages, provider=provider, **kw)   # the actual REAL model call
    dt = time.perf_counter() - t0                   # seconds elapsed
    cost = call_cost(out["in_tokens"], out["out_tokens"], TIER_OF[provider])
    log.info("provider=%s in=%d out=%d cost=$%.6f latency=%.3fs",
             provider, out["in_tokens"], out["out_tokens"], cost, dt)
    return out

out = chat_logged(messages, provider="gemini")
print("\nreply:", out["text"])

# You get one INFO line with the provider, real token counts, the computed cost,
# and the real latency, then the model's reply. Switch provider to "groq" or
# "gemini" (with a key in .env) and the same line reports that provider's numbers.

2026-08-31 21:14:29,782 | INFO | AFC is enabled with max remote calls: 10.
2026-08-31 21:14:29,783 | WARNING | Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
2026-08-31 21:14:30,819 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash-lite:generateContent "HTTP/1.1 200 OK"
2026-08-31 21:14:30,822 | INFO | provider=gemini in=26 out=36 cost=$0.000004 latency=1.897s



reply: An invoice is a commercial document issued by a seller to a buyer, itemizing products or services provided, agreed-upon prices, and the total payment due by a specific date.


## 5. Streaming (the responsive-feel pattern)

In [12]:
# Streaming yields tokens as they are produced. Shape of the real Groq call:
#
#   stream = client.chat.completions.create(model=..., messages=..., stream=True)
#   for chunk in stream:
#       piece = chunk.choices[0].delta.content or ""
#       print(piece, end="", flush=True)
#
# Offline, here is the same user-visible effect over the reply we already have:
import sys, time
for word in out["text"].split():
    sys.stdout.write(word + " "); sys.stdout.flush(); time.sleep(0.04)
print()

# Same final text either way; streaming just shows it arriving instead of waiting.

An invoice is a commercial document issued by a seller to a buyer, itemizing products or services provided, agreed-upon prices, and the total payment due by a specific date. 


## The packaged wrapper: `llm.py`

The wrapper you just built is shipped in this folder as `llm.py`, trimmed to the three real providers (gemini, groq, ollama) and one `chat(messages)` call. Every later class imports it instead of rewriting the plumbing:

```python
import llm                       # reads PROVIDER and keys from .env
reply = llm.chat([{"role": "user", "content": "What is an invoice?"}])
```

Set `PROVIDER` and the keys in a `.env` at the project root (copy `.env.example`). Default provider is `ollama` (free, local); switch to `groq` or `gemini` by editing `.env`.

## Recap

You built a message list with a real system prompt, estimated tokens and cost, routed one `chat()` across four providers (echo, ollama, groq, gemini), and logged provider, tokens, cost, and latency on every call. The micro-assignment turns this into a small Streamlit app with a provider switch. Next class: prompt engineering for production.

**Companion files in this folder:** `app.py` is the full Streamlit build (run `streamlit run app.py`), and `logging_basics.ipynb` is a focused tour of the `logging` module.